# Stage 14 — RL Policy: PPO Agent
**Dashboard page:** RL Policy
**Algorithm:** PPO (stable-baselines3) · multiplier ∈ [0.5, 1.5] on rule-based ROQ
**State:** [stock_ratio, in_transit×2, demand_std_norm, coverage_months, abc_int, tier_int, cycle_fraction]
**Reward:** -(holding + 5×stockout + ordering) / unit_value

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
rl = load("rl_policy.parquet")
print(f"RL policy: {len(rl):,} SKUs")
print(f"Columns  : {rl.columns.tolist()}")
print()
mult = rl["rl_multiplier"]
print(f"Multiplier stats:")
print(f"  Mean   : {mult.mean():.4f}")
print(f"  Median : {mult.median():.4f}")
print(f"  Std    : {mult.std():.4f}")
print(f"  Min    : {mult.min():.4f}")
print(f"  Max    : {mult.max():.4f}")
print()
print(f"RL orders MORE than rule-based (>1.0): {(mult>1.0).sum():,} ({(mult>1.0).mean()*100:.1f}%)")
print(f"RL orders LESS than rule-based (<1.0): {(mult<1.0).sum():,} ({(mult<1.0).mean()*100:.1f}%)")
print(f"RL-flagged (|mult-1|>0.50): {rl['rl_flag'].sum() if 'rl_flag' in rl.columns else 'N/A'}")


## RL Multiplier Distribution

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(13,5))
mult.hist(bins=80,ax=axes[0],color=PALETTE[4],edgecolor="white",alpha=0.8)
axes[0].axvline(1.0,color=PALETTE[1],ls="--",lw=2,label="Rule-based baseline (x1.0)")
axes[0].axvline(0.5,color="gray",ls=":",lw=1.5,label="Lower bound (x0.5)")
axes[0].axvline(1.5,color="gray",ls=":",lw=1.5,label="Upper bound (x1.5)")
axes[0].set_title("RL Order Multiplier Distribution
(clipped ±50% from rule-based ROQ)")
axes[0].set_xlabel("Multiplier"); axes[0].legend(fontsize=8)

# Multiplier by policy tier
tier_col = "policy_tier"
if tier_col in rl.columns:
    tiers = [t for t in ["critical","managed","watch","rationalise"] if t in rl[tier_col].values]
    boxes = [rl.loc[rl[tier_col]==t,"rl_multiplier"].dropna().tolist() for t in tiers]
    bp = axes[1].boxplot(boxes,labels=tiers,patch_artist=True,showfliers=False)
    for patch,t in zip(bp["boxes"],tiers):
        patch.set_facecolor(TIER_COLORS.get(t,PALETTE[0])); patch.set_alpha(0.6)
    axes[1].axhline(1.0,color=PALETTE[1],ls="--",lw=1.5,label="Baseline")
    axes[1].set_title("RL Multiplier by Policy Tier
(Critical→orders more; Rationalise→orders less)")
    axes[1].set_ylabel("Multiplier"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## RL Qty vs Rule-Based ROQ

In [ ]:
valid = rl[["roq","rl_recommended_qty"]].dropna()
valid = valid[(valid["roq"]>0)&(valid["rl_recommended_qty"]>0)]
lim = min(valid["roq"].quantile(0.95),valid["rl_recommended_qty"].quantile(0.95))

fig,axes = plt.subplots(1,2,figsize=(13,5))
axes[0].scatter(valid["roq"].clip(upper=lim),valid["rl_recommended_qty"].clip(upper=lim),
                alpha=0.2,s=12,color=PALETTE[4])
axes[0].plot([0,lim],[0,lim],"--",color="gray",alpha=0.5,label="RL=Rule-based")
axes[0].set_title("Rule-based ROQ vs RL Recommended Qty
(above diagonal: RL orders more)")
axes[0].set_xlabel("Rule-based ROQ"); axes[0].set_ylabel("RL Recommended Qty"); axes[0].legend()

pct_change = ((valid["rl_recommended_qty"]-valid["roq"])/valid["roq"]*100).clip(-60,60)
axes[1].hist(pct_change,bins=70,color=PALETTE[0],edgecolor="white",alpha=0.8)
axes[1].axvline(0,color=PALETTE[1],ls="--",lw=1.5,label="No change (0%)")
axes[1].set_title("RL Qty Adjustment (% change from rule-based ROQ, clipped ±60%)")
axes[1].set_xlabel("% change from ROQ"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Median adjustment: {pct_change.median():+.1f}%")
print(f"SKUs within ±5% of baseline: {(pct_change.abs()<5).sum():,} ({(pct_change.abs()<5).mean()*100:.1f}%)")


## Flagged SKUs (require human review)

In [ ]:
flag_col = "rl_flag" if "rl_flag" in rl.columns else None
if flag_col:
    flagged = rl[rl[flag_col]==True] if rl[flag_col].dtype==bool else rl[rl[flag_col]==1]
    print(f"RL-flagged SKUs: {len(flagged):,} (|multiplier-1.0|>0.50 before clipping)")
    if len(flagged):
        display_cols = [c for c in ["material_9","description","rl_multiplier",
            "rl_recommended_qty","roq","stock_status","policy_tier"] if c in flagged.columns]
        print(flagged[display_cols].head(20).to_string(index=False))
        print("
Flagged SKUs by policy tier:")
        print(flagged["policy_tier"].value_counts().to_string())
else:
    high_mult = rl[(rl["rl_multiplier"]<0.55)|(rl["rl_multiplier"]>1.45)]
    print(f"SKUs near bounds (mult<0.55 or >1.45): {len(high_mult):,}")
    print(high_mult[["material_9","description","rl_multiplier","policy_tier"]].head(20).to_string(index=False))


## RL Constraint: Why ±50%?

The `RL_POLICY_BAND = 0.50` hard guardrail prevents the agent from overriding the
rule-based EOQ/cover-period policy by more than 50%.

**Rationale:**
- EOQ has sound economic foundations — large deviations require human review
- RL training with noisy zero-inflated Poisson demand can over-fit to extreme episodes
- Any SKU near the bound (≈0.5 or ≈1.5) signals the rule-based policy itself needs revisiting

**Monitoring:** `rl_flag=True` SKUs are surfaced in the dashboard for the planning analyst.